[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/reflections/tilings.ipynb)

# Tilings of three geometries

Three mirrors at angles of a half turn divided by 2, 3 and $n$ cut out a triangle whose angles sum to more than a half turn for $n = 5$, to exactly a half turn for $n = 6$, and to less for $n = 7$. Reflected in one another, the mirrors tile the sphere, the flat plane and the hyperbolic plane, and one code path draws all three. The angles decide what the direction normal to the first two mirrors squares to, plus one, zero or minus one, and that is the signature of the algebra. The same rotor that turns the sphere then shifts the flat plane and boosts the hyperbolic one.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline

import numpy as np
from IPython.display import Image, display

from numga import Algebra, NumpyContext, stack
from examples.animation import save_animation
from examples.math.reflections import render

# The angles at which the mirrors meet: the first and the second, the first and the third, and the second
# and the third, which differs between the three triangles.
FIRST_SECOND = np.pi / 2
FIRST_THIRD = np.pi / 3
SECOND_THIRD = np.pi / np.array([5, 6, 7])
# The algebra of each geometry, by what the normal to the first two mirrors squares to.
SIGNATURES = {1: "x+y+z+", 0: "x+y+w0", -1: "x+y+t-"}
# The flat plane has no size of its own; the third mirror's offset sets the size of its tiles.
FLAT_OFFSET = 0.5
RESOLUTION = 300
ROUNDS = 60
FRAMES = 60
FRAME_MS = 60
# How far the views move over the animation: an angle, a distance, a rapidity.
TRAVEL = 1.2

## 1. The angles decide the geometry

The angles of the triangle sum to more than a half turn, to exactly one, or to less. As for the sphere, the first two mirror vectors are $x$ and $x$ turned in the $xy$ plane, and the third has a part in their plane, from its pairings $-\cos(\pi / m)$ with them through the reciprocal vectors of the plane, and a part along the plane's normal, its dual. For the third to have unit length, the normal must square to plus one where the part in the plane is shorter than one, to zero where it has length one, and to minus one where it is longer: the angle sum decides the signature of the algebra. On the flat plane the normal adds no length, and its part sets the size of the tiles.

In [ ]:
def geometry(second_third: float):
    """The algebra's multivectors, the sign of what the normal to the first two mirrors squares to, the
    normal, and the three mirror vectors of the triangle whose second and third mirrors meet at the given
    angle."""
    # The angles sum to more than a half turn, to one, or to less.
    curvature = int(np.sign(np.round(FIRST_SECOND + FIRST_THIRD + second_third - np.pi, 12)))
    mv = NumpyContext(Algebra(SIGNATURES[curvature])).multivector
    first = mv.x                                                                # [] Vector
    second = ((mv.x ^ mv.y) * (-(np.pi - FIRST_SECOND) / 2)).exp() >> first    # [] Vector
    # The third mirror's part in the plane of the first two, from its pairings with them through the
    # reciprocal vectors in that plane, and its part along the plane's normal, which squares to the
    # curvature: what the unit length leaves on the sphere and the hyperbolic plane, any part on the flat
    # plane, where the normal adds no length.
    plane = first ^ second                                                      # [] Bivector
    in_plane = -np.cos(FIRST_THIRD) * (second | plane.inverse()) - np.cos(second_third) * (plane.inverse() | first)   # [] Vector
    normal = plane.normalized().dual()                                         # [] Vector
    height = ((1 - in_plane.squared()) / curvature).square_root() if curvature else FLAT_OFFSET
    third = in_plane + height * normal                                          # [] Vector
    return mv, curvature, normal, stack([first, second, third])


geometries = [geometry(angle) for angle in SECOND_THIRD]

## 2. Points, sides and reflections

A point is a bivector, the dual of a direction: the wedge of two lines through it. Its side of a mirror is the sign of its regressive product with the mirror vector, which needs no metric, and reflecting it in the mirror is the sandwich with the mirror vector, negated; both read the same in the three geometries. The view is a disk, mapped to each geometry conformally: stereographically onto the sphere, to scale onto the plane, and as the Poincaré disk onto the hyperbolic plane, three cases of one formula in the square of the normal. Folding each pixel's point into the triangle and counting its reflections colours the tiles.

In [ ]:
def fold(mv, points, mirrors, rounds: int):
    """The points reflected round after round in each mirror they lie on the wrong side of, and how
    many reflections each took."""
    flips = mv.scalar(np.zeros(points.shape + (1,)))                            # [...] Scalar
    for _ in range(rounds):
        for index in range(mirrors.shape[0]):
            side = mirrors[index] & points                                      # [...] Scalar
            # One on the wrong side of the mirror, zero on the right side.
            wrong = (side.abs() - side) / (2 * side.abs())                      # [...] Scalar
            points = points + wrong * (-(mirrors[index] >> points) - points)
            flips = flips + wrong
    return points, flips


def view(mv, curvature: int, normal):
    """The disk of pixels as points of the geometry: its direction, conformally mapped, dualized."""
    u, v = render.disk(RESOLUTION)
    return (mv.x * (2 * u) + mv.y * (2 * v) + normal * (1 - curvature * (u**2 + v**2))).dual()


def moving(frames: int):
    """Per frame, every geometry moved along x by the rotor of the plane of x and the normal, which
    turns the sphere, shifts the plane and boosts the hyperbolic plane: its folded points, their
    reflection counts and its mirrors."""
    views = [view(mv, curvature, normal) for mv, curvature, normal, _ in geometries]
    for travel in np.linspace(0.0, TRAVEL, frames):
        yield [fold(mv, ((mv.x ^ normal) * (-travel / 2)).exp() >> points, mirrors, ROUNDS) + (mirrors,)
               for (mv, _, normal, mirrors), points in zip(geometries, views)]

## 3. The tilings

Sphere, plane and hyperbolic plane, moving along $x$ together.

In [ ]:
display(Image(filename=save_animation(render.animate_tilings(moving(FRAMES)), "tilings", FRAME_MS)))

In [ ]:
# checks
# The three triangles fall in the three signatures; the plane of x and the normal squares to minus
# the normal's square, so the same rotor turns, shifts or boosts; the mirror vectors pair to -cos of
# the angle between their mirrors; every folded point inside the disk lies on the right side of all
# three mirrors.
assert [curvature for _, curvature, _, _ in geometries] == [1, 0, -1]
u, v = render.disk(RESOLUTION)
inside = u**2 + v**2 < 0.98**2
for (mv, curvature, normal, mirrors), angle in zip(geometries, SECOND_THIRD):
    np.testing.assert_allclose((mv.x ^ normal).squared().to_array(), -curvature, atol=1e-12)
    first, second, third = mirrors[0], mirrors[1], mirrors[2]
    for one, other, between in ((first, second, FIRST_SECOND), (first, third, FIRST_THIRD), (second, third, angle)):
        np.testing.assert_allclose((one | other).to_array(), -np.cos(between), atol=1e-12)
for step in moving(2):
    for (folded, flips, mirrors) in step:
        sides = (mirrors & folded[..., None]).to_array()[inside]
        assert np.isfinite(sides).all() and (sides > -1e-9).all()